# 01 — Extracción de datos
## Contratos Menores · Ayuntamiento de Vilagarcía de Arousa

El primer paso del proyecto es descargar el dataset y entender qué contiene.

**Fuente:** https://sede.vilagarcia.gal/dcsv/C0G6N1Z26P99PL6H

## Paso 0 — Instalar dependencias
Ejecuta esta celda solo la primera vez. Después reinicia el kernel (Kernel → Restart).

In [ ]:
%pip install --prefer-binary -q -r ../requirements.txt

## Paso 1 — Descargar el fichero CSV

Descargamos el dataset desde la sede electrónica y lo guardamos en `data/raw/`.  
Si ya existe no lo volvemos a descargar (idempotencia básica).

In [ ]:
import requests
from pathlib import Path

URL = "https://sede.vilagarcia.gal/dcsv/C0G6N1Z26P99PL6H"
DESTINO = Path("../data/raw/contratos_menores.csv")

# Solo descargamos si el fichero no existe todavía
if DESTINO.exists():
    print(f"El fichero ya existe: {DESTINO}")
else:
    print("Descargando...")
    respuesta = requests.get(URL, timeout=30)
    respuesta.raise_for_status()          # lanza error si el servidor falla
    DESTINO.write_bytes(respuesta.content)
    print(f"Guardado en: {DESTINO}  ({len(respuesta.content):,} bytes)")

## Paso 2 — Cargar el CSV con pandas y ver su estructura

Antes de tocar nada, entendemos qué hay dentro.

In [ ]:
import pandas as pd

# Probamos primero con separador ; (habitual en CSVs españoles)
# y codificación latin-1 (habitual en ficheros de administraciones públicas)
df = pd.read_csv(DESTINO, sep=";", encoding="latin-1")

print(f"Filas:    {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"\nNombres de columnas:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
# Primeras 5 filas — ver cómo son los datos reales
df.head()

## Paso 3 — Primera revisión de calidad

Cuántos valores vacíos hay en cada columna y qué tipo de dato tiene cada una.

In [ ]:
# Tipo de dato y cuántos nulos tiene cada columna
resumen = pd.DataFrame({
    "tipo":  df.dtypes,
    "nulos": df.isnull().sum(),
    "% nulos": (df.isnull().mean() * 100).round(1)
})
resumen

## ¿Qué hacer a continuación?

Con esto ya tienes los datos en bruto cargados. El siguiente paso (notebook 02) será:

1. **Limpiar** los datos: fechas, importes, valores vacíos.
2. **Normalizar** los nombres de columnas.
3. Preparar un DataFrame limpio listo para cargar en Snowflake.

> **Tarea:** Ejecuta las celdas anteriores y anota qué columnas encuentras, qué tipos tienen
> y si ves algo raro (fechas con formato extraño, importes como texto, columnas sin nombre...).
> Esas observaciones guiarán la fase de transformación.